In [14]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [15]:
# Carregamento e visualização do nosso dataframe
df = pd.read_csv("titanic_v2.csv")
df.head()

,passenger,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [16]:
# Visualização da informação para posteriormente proceder ao tratamento de dados do dataframe.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   passenger  891 non-null    int64  
 1   survived   891 non-null    int64  
 2   pclass     891 non-null    int64  
 3   name       891 non-null    object 
 4   sex        891 non-null    object 
 5   age        714 non-null    float64
 6   sibsp      891 non-null    int64  
 7   parch      891 non-null    int64  
 8   ticket     891 non-null    object 
 9   fare       891 non-null    float64
 10  cabin      204 non-null    object 
 11  embarked   889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [17]:
# Remoção da coluna "cabin" por cerca de 80% dos seus valores serem nulos.
df = df.drop(["cabin"], axis=1)

In [18]:
# Preencher os valores nulos da coluna "age" com a mediana da mesma.
df["age"] = df["age"].fillna(df["age"].median())

In [19]:
# Ao fazer este dropna(), removo 2 linhas da coluna "embarked" que apresentavam valores nulos.
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   passenger  889 non-null    int64  
 1   survived   889 non-null    int64  
 2   pclass     889 non-null    int64  
 3   name       889 non-null    object 
 4   sex        889 non-null    object 
 5   age        889 non-null    float64
 6   sibsp      889 non-null    int64  
 7   parch      889 non-null    int64  
 8   ticket     889 non-null    object 
 9   fare       889 non-null    float64
 10  embarked   889 non-null    object 
dtypes: float64(2), int64(5), object(4)
memory usage: 83.3+ KB


In [20]:
# Mapeamento das variáveis categóricas para as transformar em variáveis numéricas.
df["sex"] = df["sex"].map({
    "male": 0,
    "female": 1
})

In [21]:
df = pd.get_dummies(df, columns=["embarked"], dtype=int)
df.head()

,passenger,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,embarked_C,embarked_Q,embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,0,0,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,1,0,0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,0,0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,0,0,1
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,0,0,1


In [22]:
# Treinar o modelo
df_train = df.copy().sample(frac=0.8, random_state=42)
y_train = df_train["survived"]
X_train = df_train[["pclass", "sex"]]

X_train = sm.add_constant(X_train, prepend=False)
model = sm.Logit(y_train,X_train)

m_fitted = model.fit()
print(m_fitted.summary())


Optimization terminated successfully.
         Current function value: 0.455203
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               survived   No. Observations:                  711
Model:                          Logit   Df Residuals:                      708
Method:                           MLE   Df Model:                            2
Date:                Wed, 11 Mar 2026   Pseudo R-squ.:                  0.3204
Time:                        22:34:10   Log-Likelihood:                -323.65
converged:                       True   LL-Null:                       -476.24
Covariance Type:            nonrobust   LLR p-value:                 5.400e-67
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
pclass        -1.0694      0.122     -8.746      0.000      -1.309      -0.830
sex            2.7131      0.

In [23]:
# Teste do modelo
df_test = df.copy().drop(df_train.index)
y_test = df_test["survived"]
X_test = df_test[["pclass", "sex"]]

X_test = sm.add_constant(X_test,prepend=False)
pred = m_fitted.predict(X_test)
pred.head()

1     0.926187
4     0.089285
13    0.089285
14    0.596454
20    0.222184
dtype: float64

In [24]:
predicoes = (pred >= 0.5).astype(int)

In [25]:
pd.crosstab(
    y_test,
    predicoes,
    rownames=["Real"], 
    colnames=["Previsto"])

Previsto,0,1
Real,,
0,101,16
1,22,39


In [26]:
# Taxa de acerto total
accuracy = np.mean(y_test == predicoes)
print("Accuracy:",round(accuracy*100,2), "%")

Accuracy: 78.65 %
